# 03 Recipe Preprocessing

## Purpose

Clean and preprocess the recipe metadata tables so that they can be used in downstream
feature engineering, modelling, and dashboard display.

This notebook builds on the earlier audit and interaction-cleaning stages.

## Objectives

- load `RAW_recipes.csv` and `PP_recipes.csv`
- confirm the recipe-side join assumptions from the audit
- parse list-like columns safely
- standardise recipe identifiers
- clean basic text and numeric fields
- derive compact recipe features from readable and tokenised metadata
- join `RAW_recipes` and `PP_recipes`
- measure recipe-table coverage after the join
- save cleaned recipe outputs for later phases

## Prior findings carried into this phase

`01_dataset_audit.ipynb` established that:

- `raw_recipes` contains 231,637 rows and acts as the complete recipe reference table
- `pp_recipes` contains 178,265 rows and does not cover all recipes
- join coverage from interactions to `raw_recipes` is 100%
- join coverage from interactions to `pp_recipes` is only 76.9588%
- recipe joins must therefore use `recipe_id` rather than recipe name

`02_interaction_cleaning.ipynb` established that:

- cleaned interaction datasets have already been saved for downstream use
- later modelling stages will require recipe-side metadata aligned to the cleaned interaction data

This notebook therefore treats `RAW_recipes.csv` as the primary recipe metadata source
and enriches it with structured fields from `PP_recipes.csv`.

In [32]:
from __future__ import annotations

import ast
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [33]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

In [34]:
PROJECT_ROOT = Path.cwd().resolve().parents[0]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [35]:
from src.paths import (
    RAW_RECIPES_PATH,
    PP_RECIPES_PATH,
    PROCESSED_DIR,
    INTERIM_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    ensure_directories,
)

ensure_directories()
print("Output directories are ready.")

Output directories are ready.


## 1. Loading recipe datasets

Load the two recipe-side files required for this phase:

- `RAW_recipes.csv`
- `PP_recipes.csv`

`RAW_recipes.csv` contains the readable metadata and full recipe catalogue.

`PP_recipes.csv` contains structured tokenised fields and compact machine-oriented
attributes, but with lower coverage than the raw recipe table.

In [36]:
raw_recipes = pd.read_csv(RAW_RECIPES_PATH)
pp_recipes = pd.read_csv(PP_RECIPES_PATH)

print("Raw recipes shape:", raw_recipes.shape)
print("PP recipes shape:", pp_recipes.shape)

Raw recipes shape: (231637, 12)
PP recipes shape: (178265, 8)


In [37]:
display(raw_recipes.head(3))
display(pp_recipes.head(3))

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'occasion', 'north-american', 'side-dishes', 'vegetables', 'mexican', 'easy', 'fall', 'holiday-event',...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'depending on size of squash , cut into half or fourths', 'remove seeds', 'for spicy squash , drizzle olive oil or melted butter over each cut squash piec...","autumn is my favorite time of year to cook! this recipe \r\ncan be prepared either spicy or sweet, your choice!\r\ntwo of my posted mexican-inspired seasoning mix recipes are offered as suggestions.","['winter squash', 'mexican seasoning', 'mixed spice', 'honey', 'butter', 'olive oil', 'salt']",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'occasion', 'north-american', 'breakfast', 'main-dish', 'pork', 'american', 'oven', 'easy', 'kid-frien...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough into the bottom and sides of a 12 inch pizza pan', 'bake for 5 minutes until set but not browned', 'cut sausage into small pieces', 'whisk eggs and m...",this recipe calls for the crust to be prebaked a bit before adding ingredients. feel free to change sausage to ham or bacon. this warms well in the microwave for those late risers.,"['prepared pizza crust', 'sausage patty', 'eggs', 'milk', 'salt and pepper', 'cheese']",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'main-dish', 'chili', 'crock-pot-slow-cooker', 'dietary', 'equipment', '4-hours-or-less']","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add chopped onions to ground beef when almost brown and sautee until wilted', 'add all other ingredients', 'add kidney beans if you like beans in your chili', '...",this modified version of 'mom's' chili was a hit at our 2004 christmas party. we made an extra large pot to have some left to freeze but it never made it to the freezer. it was a favorite by all. ...,"['ground beef', 'yellow onions', 'diced tomatoes', 'tomato paste', 'tomato soup', 'rotel tomatoes', 'kidney beans', 'water', 'chili powder', 'ground cumin', 'salt', 'lettuce', 'cheddar cheese']",13


,id,i,name_tokens,ingredient_tokens,steps_tokens,techniques,calorie_level,ingredient_ids
0,424415,23,"[40480, 37229, 2911, 1019, 249, 6878, 6878, 2839, 1781, 40481]","[[2911, 1019, 249, 6878], [1353], [6953], [15341, 3261], [2056, 857, 643, 1631, 20480]]","[40480, 40482, 21662, 481, 6878, 500, 246, 1614, 1911, 10757, 240, 674, 9933, 8400, 40478, 40482, 1082, 589, 16126, 500, 481, 6878, 2839, 1781, 5024, 240, 488, 13770, 485, 23667, 40478, 40482, 123...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]",0,"[389, 7655, 6270, 1527, 3406]"
1,146223,96900,"[40480, 18376, 7056, 246, 1531, 2032, 40481]","[[17918], [25916], [2507, 6444], [8467, 1179], [8780], [6812], [4370, 2653, 18376], [2654, 5581, 34904, 5940], [15341], [10848], [21447, 7869], [6953]]","[40480, 40482, 729, 2525, 10906, 485, 43, 8393, 40478, 40482, 23667, 17918, 240, 25916, 240, 2507, 6444, 488, 8467, 1179, 40478, 40482, 4846, 6737, 8780, 488, 7087, 862, 40478, 40482, 3336, 666, 4...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]",0,"[2683, 4969, 800, 5298, 840, 2499, 6632, 7022, 1511, 3248, 4964, 6270]"
2,312329,120056,"[40480, 21044, 16954, 8294, 556, 10837, 40481]","[[5867, 24176], [1353], [6953], [1301, 11332], [21453, 8361], [25845, 8111, 11332], [23488, 8361], [37754, 10734], [652, 25, 3035, 11959, 10734], [10837], [19811, 21137, 556, 20323, 15022, 296, 7,...","[40480, 40482, 8240, 481, 24176, 296, 1353, 666, 246, 1719, 5082, 40478, 40482, 4846, 481, 21298, 240, 37754, 10734, 240, 652, 25, 3035, 11959, 240, 296, 10837, 485, 481, 5082, 674, 2030, 485, 246...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,"[1257, 7655, 6270, 590, 5024, 1119, 4883, 6696, 7946, 5648, 7239, 7705, 7594, 1168, 2683]"


## 2. Confirming recipe-table assumptions from the audit

The earlier audit showed that:

- `raw_recipes` is the full recipe metadata table
- `pp_recipes` is smaller and should not be treated as the sole recipe source
- recipe joins should always be based on `recipe_id`

This section reconfirms those assumptions before cleaning begins.

In [38]:
raw_recipe_ids = set(raw_recipes["id"].dropna().unique())
pp_recipe_ids = set(pp_recipes["id"].dropna().unique())

recipe_table_comparison = pd.DataFrame([
    {
        "table": "raw_recipes",
        "rows": int(len(raw_recipes)),
        "unique_recipe_ids": int(raw_recipes["id"].nunique()),
    },
    {
        "table": "pp_recipes",
        "rows": int(len(pp_recipes)),
        "unique_recipe_ids": int(pp_recipes["id"].nunique()),
    },
    {
        "table": "recipes_in_both_tables",
        "rows": int(len(raw_recipe_ids & pp_recipe_ids)),
        "unique_recipe_ids": int(len(raw_recipe_ids & pp_recipe_ids)),
    },
    {
        "table": "raw_only_recipe_ids",
        "rows": int(len(raw_recipe_ids - pp_recipe_ids)),
        "unique_recipe_ids": int(len(raw_recipe_ids - pp_recipe_ids)),
    },
])

display(recipe_table_comparison)

,table,rows,unique_recipe_ids
0,raw_recipes,231637,231637
1,pp_recipes,178265,178265
2,recipes_in_both_tables,178265,178265
3,raw_only_recipe_ids,53372,53372


In [39]:
recipe_join_coverage = pd.DataFrame([
    {
        "comparison": "raw_recipes vs pp_recipes",
        "raw_recipe_ids": int(len(raw_recipe_ids)),
        "pp_recipe_ids": int(len(pp_recipe_ids)),
        "matched_ids": int(len(raw_recipe_ids & pp_recipe_ids)),
        "missing_pp_ids_from_raw": int(len(raw_recipe_ids - pp_recipe_ids)),
        "coverage_pct_of_raw": round(len(raw_recipe_ids & pp_recipe_ids) / len(raw_recipe_ids) * 100, 4),
    }
])

display(recipe_join_coverage)
print("Sample recipe IDs present in raw_recipes but missing in pp_recipes:")
print(sorted(list(raw_recipe_ids - pp_recipe_ids))[:20])

,comparison,raw_recipe_ids,pp_recipe_ids,matched_ids,missing_pp_ids_from_raw,coverage_pct_of_raw
0,raw_recipes vs pp_recipes,231637,178265,178265,53372,76.9588


Sample recipe IDs present in raw_recipes but missing in pp_recipes:
[np.int64(39), np.int64(41), np.int64(43), np.int64(48), np.int64(50), np.int64(55), np.int64(67), np.int64(81), np.int64(115), np.int64(118), np.int64(119), np.int64(122), np.int64(133), np.int64(135), np.int64(141), np.int64(144), np.int64(146), np.int64(147), np.int64(148), np.int64(149)]


### Observation notes

* `RAW_recipes.csv` is confirmed as the larger recipe catalogue, containing **231,637** unique recipe IDs.
* `PP_recipes.csv` contains **178,265** unique recipe IDs and is therefore only a **subset** of the recipes available in `RAW_recipes.csv`.
* All recipe IDs found in `PP_recipes.csv` are also present in `RAW_recipes.csv`, since the number of matched IDs is exactly **178,265**.
* `RAW_recipes.csv` contains **53,372** recipe IDs that do not appear in `PP_recipes.csv`.
* This means `PP_recipes.csv` covers only **76.9588%** of the recipes in `RAW_recipes.csv`, so relying only on `PP_recipes.csv` would exclude a substantial portion of the available recipe catalogue.
* The recipe preprocessing strategy should therefore:

  * use `raw_recipes` as the base recipe table
  * enrich recipes with `pp_recipes` features where available
  * preserve recipes that do not have corresponding preprocessed/tokenised metadata in `pp_recipes`

## 3. Keeping the required recipe columns

Only the fields needed for recipe preprocessing and later modelling are retained.

In [40]:
raw_required_columns = [
    "id",
    "name",
    "minutes",
    "contributor_id",
    "submitted",
    "tags",
    "nutrition",
    "n_steps",
    "steps",
    "description",
    "ingredients",
    "n_ingredients",
]

pp_required_columns = [
    "id",
    "i",
    "name_tokens",
    "ingredient_tokens",
    "steps_tokens",
    "techniques",
    "calorie_level",
    "ingredient_ids",
]

raw_df = raw_recipes[raw_required_columns].copy()
pp_df = pp_recipes[pp_required_columns].copy()

print("Raw recipe working shape:", raw_df.shape)
print("PP recipe working shape:", pp_df.shape)

Raw recipe working shape: (231637, 12)
PP recipe working shape: (178265, 8)


In [41]:
display(raw_df.head(2))
display(pp_df.head(2))

,id,name,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,137739,arriba baked winter squash mexican style,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'occasion', 'north-american', 'side-dishes', 'vegetables', 'mexican', 'easy', 'fall', 'holiday-event',...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'depending on size of squash , cut into half or fourths', 'remove seeds', 'for spicy squash , drizzle olive oil or melted butter over each cut squash piec...","autumn is my favorite time of year to cook! this recipe \r\ncan be prepared either spicy or sweet, your choice!\r\ntwo of my posted mexican-inspired seasoning mix recipes are offered as suggestions.","['winter squash', 'mexican seasoning', 'mixed spice', 'honey', 'butter', 'olive oil', 'salt']",7
1,31490,a bit different breakfast pizza,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'occasion', 'north-american', 'breakfast', 'main-dish', 'pork', 'american', 'oven', 'easy', 'kid-frien...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough into the bottom and sides of a 12 inch pizza pan', 'bake for 5 minutes until set but not browned', 'cut sausage into small pieces', 'whisk eggs and m...",this recipe calls for the crust to be prebaked a bit before adding ingredients. feel free to change sausage to ham or bacon. this warms well in the microwave for those late risers.,"['prepared pizza crust', 'sausage patty', 'eggs', 'milk', 'salt and pepper', 'cheese']",6


,id,i,name_tokens,ingredient_tokens,steps_tokens,techniques,calorie_level,ingredient_ids
0,424415,23,"[40480, 37229, 2911, 1019, 249, 6878, 6878, 2839, 1781, 40481]","[[2911, 1019, 249, 6878], [1353], [6953], [15341, 3261], [2056, 857, 643, 1631, 20480]]","[40480, 40482, 21662, 481, 6878, 500, 246, 1614, 1911, 10757, 240, 674, 9933, 8400, 40478, 40482, 1082, 589, 16126, 500, 481, 6878, 2839, 1781, 5024, 240, 488, 13770, 485, 23667, 40478, 40482, 123...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]",0,"[389, 7655, 6270, 1527, 3406]"
1,146223,96900,"[40480, 18376, 7056, 246, 1531, 2032, 40481]","[[17918], [25916], [2507, 6444], [8467, 1179], [8780], [6812], [4370, 2653, 18376], [2654, 5581, 34904, 5940], [15341], [10848], [21447, 7869], [6953]]","[40480, 40482, 729, 2525, 10906, 485, 43, 8393, 40478, 40482, 23667, 17918, 240, 25916, 240, 2507, 6444, 488, 8467, 1179, 40478, 40482, 4846, 6737, 8780, 488, 7087, 862, 40478, 40482, 3336, 666, 4...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]",0,"[2683, 4969, 800, 5298, 840, 2499, 6632, 7022, 1511, 3248, 4964, 6270]"


## 4. Standardising recipe identifiers and essential numeric fields

Recipe joins must use `recipe_id`.

This section:

- renames `id` to `recipe_id`
- checks numeric conversion for key fields
- removes rows with invalid recipe identifiers if any are found

In [42]:
raw_df = raw_df.rename(columns={"id": "recipe_id"})
pp_df = pp_df.rename(columns={"id": "recipe_id"})

for col in ["recipe_id", "minutes", "contributor_id", "n_steps", "n_ingredients"]:
    raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

for col in ["recipe_id", "i", "calorie_level"]:
    pp_df[col] = pd.to_numeric(pp_df[col], errors="coerce")

invalid_raw_id_rows = raw_df[raw_df["recipe_id"].isna()].copy()
invalid_pp_id_rows = pp_df[pp_df["recipe_id"].isna()].copy()

print("Invalid raw recipe_id rows:", len(invalid_raw_id_rows))
print("Invalid pp recipe_id rows:", len(invalid_pp_id_rows))

Invalid raw recipe_id rows: 0
Invalid pp recipe_id rows: 0


In [43]:
raw_df = raw_df.dropna(subset=["recipe_id"]).copy()
pp_df = pp_df.dropna(subset=["recipe_id"]).copy()

raw_df["recipe_id"] = raw_df["recipe_id"].astype("int64")
pp_df["recipe_id"] = pp_df["recipe_id"].astype("int64")

for col in ["minutes", "contributor_id", "n_steps", "n_ingredients"]:
    raw_df[col] = raw_df[col].astype("Int64")

for col in ["i", "calorie_level"]:
    pp_df[col] = pp_df[col].astype("Int64")

print("Raw recipe shape after ID cleaning:", raw_df.shape)
print("PP recipe shape after ID cleaning:", pp_df.shape)

Raw recipe shape after ID cleaning: (231637, 12)
PP recipe shape after ID cleaning: (178265, 8)


In [44]:
print("Raw recipe_id duplicates:", int(raw_df.duplicated(subset=["recipe_id"]).sum()))
print("PP recipe_id duplicates:", int(pp_df.duplicated(subset=["recipe_id"]).sum()))

Raw recipe_id duplicates: 0
PP recipe_id duplicates: 0


## 5. Parsing list-like columns safely

Several recipe columns are stored as Python-style list strings.

Examples include:

### In `RAW_recipes.csv`
- `tags`
- `nutrition`
- `steps`
- `ingredients`

### In `PP_recipes.csv`
- `name_tokens`
- `ingredient_tokens`
- `steps_tokens`
- `techniques`
- `ingredient_ids`

These fields are parsed into Python objects using a safe parser so that compact
derived features can be created.

In [45]:
def parse_list_like(value):
    """
    Parse a Python-style list string into a Python object.

    Rules:
    - existing list values are returned as-is
    - missing values return pd.NA
    - malformed values return pd.NA
    """
    if pd.isna(value):
        return pd.NA

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        value = value.strip()
        if value == "":
            return pd.NA
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return pd.NA

    return pd.NA

In [46]:
raw_list_columns = ["tags", "nutrition", "steps", "ingredients"]
pp_list_columns = ["name_tokens", "ingredient_tokens", "steps_tokens", "techniques", "ingredient_ids"]

for col in raw_list_columns:
    raw_df[col] = raw_df[col].apply(parse_list_like)

for col in pp_list_columns:
    pp_df[col] = pp_df[col].apply(parse_list_like)

print("List-like parsing complete.")

List-like parsing complete.


In [47]:
display(raw_df[["recipe_id", "tags", "nutrition", "steps", "ingredients"]].head(2))
display(pp_df[["recipe_id", "name_tokens", "ingredient_tokens", "steps_tokens", "techniques", "ingredient_ids"]].head(2))

,recipe_id,tags,nutrition,steps,ingredients
0,137739,"[60-minutes-or-less, time-to-make, course, main-ingredient, cuisine, preparation, occasion, north-american, side-dishes, vegetables, mexican, easy, fall, holiday-event, vegetarian, winter, dietary...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]","[make a choice and proceed with recipe, depending on size of squash , cut into half or fourths, remove seeds, for spicy squash , drizzle olive oil or melted butter over each cut squash piece, seas...","[winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt]"
1,31490,"[30-minutes-or-less, time-to-make, course, main-ingredient, cuisine, preparation, occasion, north-american, breakfast, main-dish, pork, american, oven, easy, kid-friendly, pizza, dietary, northeas...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]","[preheat oven to 425 degrees f, press dough into the bottom and sides of a 12 inch pizza pan, bake for 5 minutes until set but not browned, cut sausage into small pieces, whisk eggs and milk in a ...","[prepared pizza crust, sausage patty, eggs, milk, salt and pepper, cheese]"


,recipe_id,name_tokens,ingredient_tokens,steps_tokens,techniques,ingredient_ids
0,424415,"[40480, 37229, 2911, 1019, 249, 6878, 6878, 2839, 1781, 40481]","[[2911, 1019, 249, 6878], [1353], [6953], [15341, 3261], [2056, 857, 643, 1631, 20480]]","[40480, 40482, 21662, 481, 6878, 500, 246, 1614, 1911, 10757, 240, 674, 9933, 8400, 40478, 40482, 1082, 589, 16126, 500, 481, 6878, 2839, 1781, 5024, 240, 488, 13770, 485, 23667, 40478, 40482, 123...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]","[389, 7655, 6270, 1527, 3406]"
1,146223,"[40480, 18376, 7056, 246, 1531, 2032, 40481]","[[17918], [25916], [2507, 6444], [8467, 1179], [8780], [6812], [4370, 2653, 18376], [2654, 5581, 34904, 5940], [15341], [10848], [21447, 7869], [6953]]","[40480, 40482, 729, 2525, 10906, 485, 43, 8393, 40478, 40482, 23667, 17918, 240, 25916, 240, 2507, 6444, 488, 8467, 1179, 40478, 40482, 4846, 6737, 8780, 488, 7087, 862, 40478, 40482, 3336, 666, 4...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]","[2683, 4969, 800, 5298, 840, 2499, 6632, 7022, 1511, 3248, 4964, 6270]"


## 6. Cleaning text and date fields in the raw recipe table

Readable metadata is useful for dashboard display, interpretation, and derived features.

This section cleans:

- recipe name
- description
- submitted date

In [48]:
raw_df["name"] = raw_df["name"].astype("string").str.strip()
raw_df["name"] = raw_df["name"].replace("", pd.NA)

raw_df["description"] = raw_df["description"].astype("string").str.strip()
raw_df["description"] = raw_df["description"].replace("", pd.NA)

raw_df["submitted"] = pd.to_datetime(raw_df["submitted"], errors="coerce")

print("Missing recipe names:", int(raw_df["name"].isna().sum()))
print("Missing descriptions:", int(raw_df["description"].isna().sum()))
print("Invalid submitted dates:", int(raw_df["submitted"].isna().sum()))

Missing recipe names: 1
Missing descriptions: 4982
Invalid submitted dates: 0


## 7. Deriving compact recipe features from the raw recipe table

The raw recipe table contains human-readable but high-dimensional metadata.

Instead of carrying only raw list columns forward, compact summary features are derived.

In [49]:
def safe_len(value):
    if isinstance(value, list):
        return len(value)
    return pd.NA


def safe_numeric_list_stats(values):
    """
    Extract simple statistics from a numeric list, such as the nutrition vector.
    """
    if not isinstance(values, list) or len(values) == 0:
        return {
            "len": pd.NA,
            "sum": pd.NA,
            "mean": pd.NA,
        }

    numeric_values = pd.to_numeric(pd.Series(values), errors="coerce").dropna()

    if numeric_values.empty:
        return {
            "len": len(values),
            "sum": pd.NA,
            "mean": pd.NA,
        }

    return {
        "len": len(values),
        "sum": float(numeric_values.sum()),
        "mean": float(numeric_values.mean()),
    }

In [50]:
raw_df["name_length"] = raw_df["name"].str.len().astype("Int64")
raw_df["description_length"] = raw_df["description"].str.len().astype("Int64")

raw_df["tag_count"] = raw_df["tags"].apply(safe_len).astype("Int64")
raw_df["step_count_from_list"] = raw_df["steps"].apply(safe_len).astype("Int64")
raw_df["ingredient_count_from_list"] = raw_df["ingredients"].apply(safe_len).astype("Int64")

raw_df["has_description"] = raw_df["description"].notna().astype("int8")
raw_df["has_tags"] = raw_df["tags"].apply(lambda x: isinstance(x, list) and len(x) > 0).astype("int8")
raw_df["has_steps"] = raw_df["steps"].apply(lambda x: isinstance(x, list) and len(x) > 0).astype("int8")
raw_df["has_ingredients"] = raw_df["ingredients"].apply(lambda x: isinstance(x, list) and len(x) > 0).astype("int8")

In [51]:
nutrition_stats = raw_df["nutrition"].apply(safe_numeric_list_stats)

raw_df["nutrition_vector_length"] = nutrition_stats.apply(lambda x: x["len"]).astype("Int64")
raw_df["nutrition_sum"] = nutrition_stats.apply(lambda x: x["sum"])
raw_df["nutrition_mean"] = nutrition_stats.apply(lambda x: x["mean"])

display(
    raw_df[
        [
            "recipe_id",
            "minutes",
            "n_steps",
            "n_ingredients",
            "tag_count",
            "step_count_from_list",
            "ingredient_count_from_list",
            "nutrition_vector_length",
            "nutrition_sum",
            "nutrition_mean",
        ]
    ].head(5)
)

,recipe_id,minutes,n_steps,n_ingredients,tag_count,step_count_from_list,ingredient_count_from_list,nutrition_vector_length,nutrition_sum,nutrition_mean
0,137739,55,11,7,20,11,7,7,70.5,10.071429
1,31490,30,9,6,20,9,6,7,266.4,38.057143
2,112140,130,6,13,9,6,13,7,442.8,63.257143
3,59389,45,11,11,30,11,11,7,439.1,62.728571
4,44061,190,5,8,21,5,8,7,744.9,106.414286


## 8. Deriving compact recipe features from the preprocessed recipe table

The preprocessed recipe table contains structured machine-oriented metadata.

These fields are converted into compact numeric features that are easier to use in later models.

In [52]:
pp_df["name_token_count"] = pp_df["name_tokens"].apply(safe_len).astype("Int64")
pp_df["ingredient_token_group_count"] = pp_df["ingredient_tokens"].apply(safe_len).astype("Int64")
pp_df["step_token_count"] = pp_df["steps_tokens"].apply(safe_len).astype("Int64")
pp_df["technique_vector_length"] = pp_df["techniques"].apply(safe_len).astype("Int64")
pp_df["ingredient_id_count"] = pp_df["ingredient_ids"].apply(safe_len).astype("Int64")

In [53]:
def count_active_techniques(value):
    if not isinstance(value, list):
        return pd.NA
    numeric = pd.to_numeric(pd.Series(value), errors="coerce").fillna(0)
    return int((numeric > 0).sum())

pp_df["technique_count_active"] = pp_df["techniques"].apply(count_active_techniques).astype("Int64")
pp_df["has_pp_features"] = 1

display(
    pp_df[
        [
            "recipe_id",
            "i",
            "calorie_level",
            "name_token_count",
            "ingredient_token_group_count",
            "step_token_count",
            "technique_vector_length",
            "technique_count_active",
            "ingredient_id_count",
            "has_pp_features",
        ]
    ].head(5)
)

,recipe_id,i,calorie_level,name_token_count,ingredient_token_group_count,step_token_count,technique_vector_length,technique_count_active,ingredient_id_count,has_pp_features
0,424415,23,0,10,5,94,58,3,5,1
1,146223,96900,0,7,12,117,58,8,12,1
2,312329,120056,1,7,15,88,58,5,15,1
3,74301,168258,0,4,8,114,58,3,8,1
4,76272,109030,0,9,4,55,58,3,4,1


## 9. Creating simple tag-based recipe indicators

A small number of readable binary indicators are useful for analysis and later feature engineering.

These are derived from `tags` in the raw recipe table.

In [54]:
def contains_tag(tag_list, target):
    if not isinstance(tag_list, list):
        return 0
    normalised = {str(x).strip().lower() for x in tag_list}
    return int(target.lower() in normalised)

selected_tags = [
    "breakfast",
    "lunch",
    "dinner",
    "dessert",
    "healthy",
    "easy",
    "vegetarian",
    "vegan",
]

for tag in selected_tags:
    raw_df[f"tag_{tag}"] = raw_df["tags"].apply(lambda values, t=tag: contains_tag(values, t)).astype("int8")

display(raw_df[["recipe_id"] + [f"tag_{tag}" for tag in selected_tags]].head(5))

,recipe_id,tag_breakfast,tag_lunch,tag_dinner,tag_dessert,tag_healthy,tag_easy,tag_vegetarian,tag_vegan
0,137739,0,0,0,0,0,1,1,0
1,31490,1,0,0,0,0,1,0,0
2,112140,0,0,0,0,0,0,0,0
3,59389,0,0,0,0,0,1,0,0
4,44061,0,0,0,0,0,0,1,0


## 10. Building compact cleaned recipe tables before joining

Only the main fields needed downstream are retained in the cleaned outputs.

In [55]:
raw_recipe_clean = raw_df[
    [
        "recipe_id",
        "name",
        "minutes",
        "contributor_id",
        "submitted",
        "description",
        "n_steps",
        "n_ingredients",
        "name_length",
        "description_length",
        "tag_count",
        "step_count_from_list",
        "ingredient_count_from_list",
        "nutrition_vector_length",
        "nutrition_sum",
        "nutrition_mean",
        "has_description",
        "has_tags",
        "has_steps",
        "has_ingredients",
    ]
    + [f"tag_{tag}" for tag in selected_tags]
].copy()

pp_recipe_clean = pp_df[
    [
        "recipe_id",
        "i",
        "calorie_level",
        "name_token_count",
        "ingredient_token_group_count",
        "step_token_count",
        "technique_vector_length",
        "technique_count_active",
        "ingredient_id_count",
        "has_pp_features",
    ]
].copy()

print("Clean raw recipe table shape:", raw_recipe_clean.shape)
print("Clean pp recipe table shape:", pp_recipe_clean.shape)

Clean raw recipe table shape: (231637, 28)
Clean pp recipe table shape: (178265, 10)


## 11. Joining the cleaned raw and preprocessed recipe tables

The join uses `recipe_id` and keeps all rows from the raw recipe table.

This preserves the full recipe catalogue while enriching recipes that have
matching `PP_recipes` metadata.

In [56]:
recipes_joined = raw_recipe_clean.merge(
    pp_recipe_clean,
    on="recipe_id",
    how="left",
    validate="one_to_one",
)

print("Joined recipe table shape:", recipes_joined.shape)
display(recipes_joined.head(5))

Joined recipe table shape: (231637, 37)


,recipe_id,name,minutes,contributor_id,submitted,description,n_steps,n_ingredients,name_length,description_length,tag_count,step_count_from_list,ingredient_count_from_list,nutrition_vector_length,nutrition_sum,nutrition_mean,has_description,has_tags,has_steps,has_ingredients,tag_breakfast,tag_lunch,tag_dinner,tag_dessert,tag_healthy,tag_easy,tag_vegetarian,tag_vegan,i,calorie_level,name_token_count,ingredient_token_group_count,step_token_count,technique_vector_length,technique_count_active,ingredient_id_count,has_pp_features
0,137739,arriba baked winter squash mexican style,55,47892,2005-09-16,"autumn is my favorite time of year to cook! this recipe \r\ncan be prepared either spicy or sweet, your choice!\r\ntwo of my posted mexican-inspired seasoning mix recipes are offered as suggestions.",11,7,42,194,20,11,7,7,70.5,10.071429,1,1,1,1,0,0,0,0,0,1,1,0,145702,0,9,7,165,58,3,7,1.0
1,31490,a bit different breakfast pizza,30,26278,2002-06-17,this recipe calls for the crust to be prebaked a bit before adding ingredients. feel free to change sausage to ham or bacon. this warms well in the microwave for those late risers.,9,6,32,180,20,9,6,7,266.4,38.057143,1,1,1,1,1,0,0,0,0,1,0,0,33090,0,7,6,101,58,3,6,1.0
2,112140,all in the kitchen chili,130,196586,2005-02-25,this modified version of 'mom's' chili was a hit at our 2004 christmas party. we made an extra large pot to have some left to freeze but it never made it to the freezer. it was a favorite by all. ...,6,13,25,295,9,6,13,7,442.8,63.257143,1,1,1,1,0,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN
3,59389,alouette potatoes,45,68585,2003-04-14,"this is a super easy, great tasting, make ahead side dish that looks like you spent a lot more time preparing than you actually do. plus, most everything is done in advance. the times do not refle...",11,11,18,233,30,11,11,7,439.1,62.728571,1,1,1,1,0,0,0,0,0,1,0,0,44921,1,6,11,137,58,4,11,1.0
4,44061,amish tomato ketchup for canning,190,41706,2002-10-25,"my dh's amish mother raised him on this recipe. he much prefers it over store-bought ketchup. it was a taste i had to acquire, but now my ds's also prefer this type of ketchup. enjoy!",5,8,34,183,21,5,8,7,744.9,106.414286,1,1,1,1,0,0,0,0,0,0,1,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN


In [57]:
recipes_join_summary = pd.DataFrame([
    {
        "raw_recipe_rows": int(len(raw_recipe_clean)),
        "pp_recipe_rows": int(len(pp_recipe_clean)),
        "joined_rows": int(len(recipes_joined)),
        "recipes_with_pp_features": int(recipes_joined["has_pp_features"].fillna(0).sum()),
        "recipes_without_pp_features": int(recipes_joined["has_pp_features"].isna().sum()),
        "pp_feature_coverage_pct": round(recipes_joined["has_pp_features"].fillna(0).mean() * 100, 4),
    }
])

display(recipes_join_summary)

,raw_recipe_rows,pp_recipe_rows,joined_rows,recipes_with_pp_features,recipes_without_pp_features,pp_feature_coverage_pct
0,231637,178265,231637,178265,53372,76.9588



### Observation notes

* The joined recipe table preserves the full `raw_recipes` catalogue, maintaining **231,637 recipes** after the merge.
* The join strategy correctly performs a **left join on `raw_recipes`**, ensuring no recipes are dropped during integration with `pp_recipes`.
* Structured and tokenised features from `PP_recipes.csv` are available for **178,265 recipes**, while **53,372 recipes (23.04%)** do not have corresponding preprocessed features.
* This results in a **feature coverage of 76.96%**, confirming that `PP_recipes.csv` is a substantial but incomplete enrichment source.
* Recipes without `PP_recipes` matches are still retained with missing (`NA`) feature values, which:

  * prevents unnecessary data loss
  * preserves compatibility with the full interaction dataset
* The derived `has_pp_features` indicator provides a clear mechanism to:

  * filter feature-complete subsets when required
  * analyse coverage bias in later modelling stages
* The joined dataset maintains consistency and integrity:

  * no duplicate recipe IDs
  * no invalid IDs
  * no row loss during merging
* This joined recipe table is therefore suitable as the **primary recipe metadata source** for:

  * dashboard display and exploration
  * downstream feature engineering
  * interaction–recipe joins
  * hybrid recommendation models (content + collaborative)
  * evaluation of coverage and cold-start scenarios




## 12. Checking alignment with the cleaned interaction data


This section checks how well the new joined recipe table covers the recipes used in the
cleaned interaction dataset.

In [58]:
interactions_clean = pd.read_parquet(PROCESSED_DIR / "interactions_clean.parquet")

interaction_recipe_ids = set(interactions_clean["recipe_id"].dropna().unique())
joined_recipe_ids = set(recipes_joined["recipe_id"].dropna().unique())

interaction_recipe_coverage = pd.DataFrame([
    {
        "comparison": "interactions_clean vs recipes_joined",
        "unique_recipe_ids_in_interactions": int(len(interaction_recipe_ids)),
        "unique_recipe_ids_in_joined_recipes": int(len(joined_recipe_ids)),
        "matched_ids": int(len(interaction_recipe_ids & joined_recipe_ids)),
        "missing_ids": int(len(interaction_recipe_ids - joined_recipe_ids)),
        "coverage_pct": round(len(interaction_recipe_ids & joined_recipe_ids) / len(interaction_recipe_ids) * 100, 4),
    }
])

display(interaction_recipe_coverage)

,comparison,unique_recipe_ids_in_interactions,unique_recipe_ids_in_joined_recipes,matched_ids,missing_ids,coverage_pct
0,interactions_clean vs recipes_joined,231637,231637,231637,0,100.0


## 13. Additional compact recipe statistics

These summary statistics help describe the cleaned recipe feature space for later reporting
and dashboard pages.

In [59]:
recipe_stats = pd.DataFrame([
    {"metric": "n_recipes_joined", "value": int(len(recipes_joined))},
    {"metric": "n_unique_recipe_ids", "value": int(recipes_joined["recipe_id"].nunique())},
    {"metric": "recipes_with_pp_features", "value": int(recipes_joined["has_pp_features"].fillna(0).sum())},
    {"metric": "recipes_without_pp_features", "value": int(recipes_joined["has_pp_features"].isna().sum())},
    {"metric": "median_minutes", "value": float(recipes_joined["minutes"].dropna().median())},
    {"metric": "median_n_steps", "value": float(recipes_joined["n_steps"].dropna().median())},
    {"metric": "median_n_ingredients", "value": float(recipes_joined["n_ingredients"].dropna().median())},
])

display(recipe_stats)

,metric,value
0,n_recipes_joined,231637.0
1,n_unique_recipe_ids,231637.0
2,recipes_with_pp_features,178265.0
3,recipes_without_pp_features,53372.0
4,median_minutes,40.0
5,median_n_steps,9.0
6,median_n_ingredients,9.0


In [60]:
compact_feature_nulls = pd.DataFrame({
    "column": recipes_joined.columns,
    "null_count": recipes_joined.isna().sum().values,
    "null_percentage": (recipes_joined.isna().mean().values * 100).round(4),
}).sort_values(by=["null_count", "null_percentage"], ascending=False).reset_index(drop=True)

display(compact_feature_nulls.head(20))

,column,null_count,null_percentage
0,i,53372,23.0412
1,calorie_level,53372,23.0412
2,name_token_count,53372,23.0412
3,ingredient_token_group_count,53372,23.0412
4,step_token_count,53372,23.0412
5,technique_vector_length,53372,23.0412
6,technique_count_active,53372,23.0412
7,ingredient_id_count,53372,23.0412
8,has_pp_features,53372,23.0412
9,description,4982,2.1508


## 14. Saving outputs

The cleaned recipe tables and summary outputs are saved for downstream feature engineering,
modelling, and dashboard use.

In [61]:
raw_recipe_clean.to_parquet(INTERIM_DIR / "recipes_raw_clean.parquet", index=False)
pp_recipe_clean.to_parquet(INTERIM_DIR / "recipes_pp_clean.parquet", index=False)
recipes_joined.to_parquet(PROCESSED_DIR / "recipes_joined.parquet", index=False)

recipe_table_comparison.to_csv(TABLES_DIR / "03_recipe_table_comparison.csv", index=False)
recipe_join_coverage.to_csv(TABLES_DIR / "03_recipe_join_coverage.csv", index=False)
recipes_join_summary.to_csv(TABLES_DIR / "03_recipe_join_summary.csv", index=False)
interaction_recipe_coverage.to_csv(TABLES_DIR / "03_interaction_recipe_coverage.csv", index=False)
recipe_stats.to_csv(TABLES_DIR / "03_recipe_stats.csv", index=False)
compact_feature_nulls.to_csv(TABLES_DIR / "03_recipe_feature_nulls.csv", index=False)

print("Saved recipe preprocessing outputs.")

Saved recipe preprocessing outputs.


## 15. Final decisions

The following recipe-preprocessing decisions were applied:

- `RAW_recipes.csv` was used as the base recipe metadata source
- `PP_recipes.csv` was treated as an auxiliary structured-feature table
- recipe joins were performed on `recipe_id`
- list-like columns were parsed safely
- compact numeric and binary recipe features were derived
- recipes without `PP_recipes.csv` matches were retained
- a joined recipe table was created for downstream use

The next step is to move this validated logic into `src/data/clean_recipes.py`.